# Mean Flows: From-Scratch Class-Conditional 1-NFE & Few-Step Generation

This self-contained CPU notebook demonstrates **Mean Flows** (Geng et al.,
NeurIPS 2025) using `swirl_dynamics.lib.flow_map`:

* **From-Scratch Training:** Trains `flow_map.ConditionalMeanFlowModel` with
  `flow_map.RescaledMeanFlowUNet`, `flow_map.MergeCategoricalEmbCond`, and
  `flow_map.MeanFlowTrainer` directly on synthetic class-conditional 1-D bimodal
  waveforms without any separate teacher model.
* **Backward Generation:** Generates samples from Gaussian prior $x_1$ to data
  $x_0$ via `flow_map.sample_backward_mean_flow` across 1, 2, and 4 NFE:
  $$
  x_s = x_t - (t - s)\thinspace u_\theta(x_t, t, s \mid c).
  $$


## 1. Installation & Imports


In [ ]:
!pip install -q "flax>=0.12.8" git+https://github.com/google-research/swirl-dynamics.git

In [ ]:
import functools
import os
import tempfile

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
from swirl_dynamics.lib import flow_map
from swirl_dynamics.lib import flow_matching
from swirl_dynamics.projects.distillation.mean_flow.configs import config_1d

## 2. Configuration


In [ ]:
config = config_1d.get_config()
workdir = tempfile.mkdtemp(prefix="mean_flow_demo_")

## 3. Synthetic Class-Conditional 1-D Bimodal Waveforms


In [ ]:
def make_bimodal_waveforms(num_samples, grid_size, num_classes, seed):
  rng = np.random.default_rng(seed)
  grid = np.linspace(0.0, 2.0 * np.pi, grid_size, endpoint=False)
  labels = rng.integers(0, num_classes, size=(num_samples,), dtype=np.int32)
  sign = rng.choice([-1.0, 1.0], size=(num_samples, 1))
  phase = rng.uniform(0.0, 2.0 * np.pi, size=(num_samples, 1))
  freq = (labels + 1)[:, None]
  offset = (labels - 1.0)[:, None] * 1.1
  x_0 = offset + sign * 0.8 * np.sin(freq * grid[None, :] + phase)
  x_0 = x_0[..., None].astype(np.float32)
  x_1 = rng.normal(size=x_0.shape).astype(np.float32)
  return {"x_0": x_0, "x_1": x_1, "emb:label": labels}


train_data = make_bimodal_waveforms(
    config.num_train_samples,
    config.grid_size,
    config.num_classes,
    seed=config.seed,
)
eval_data = make_bimodal_waveforms(
    config.num_eval_samples,
    config.grid_size,
    config.num_classes,
    seed=config.seed + 1,
)


def batch_iterator(dataset, batch_size, seed):
  rng = np.random.default_rng(seed)
  n = dataset["x_0"].shape[0]
  while True:
    idx = rng.integers(0, n, size=(batch_size,))
    yield {k: v[idx] for k, v in dataset.items()}

## 4. Train `ConditionalMeanFlowModel` from Scratch


In [ ]:
base_time_sampler = functools.partial(
    flow_map.logit_normal_dist,
    mean=config.logit_normal_mean,
    std=config.logit_normal_std,
)
time_sampler = functools.partial(
    flow_map.time_sampler_mean_flow,
    time_sampling=base_time_sampler,
)

net = flow_map.RescaledMeanFlowUNet(
    out_channels=config.num_channels,
    num_channels=config.num_channels_unet,
    downsample_ratio=config.downsample_ratio,
    num_blocks=config.num_blocks,
    noise_embed_dim=config.noise_embed_dim,
    use_attention=False,
    use_position_encoding=False,
    time_conditioning="t_dt",
    cond_embed_fn=flow_map.MergeCategoricalEmbCond,
    cond_embed_kwargs={
        "cond_key": "emb:label",
        "num_classes": config.num_classes,
        "features_embedding": config.features_embedding,
    },
)
mf_model = flow_map.ConditionalMeanFlowModel(
    input_shape=(config.grid_size, config.num_channels),
    cond_shape={"emb:label": ()},
    mean_flow_model=net,
    interpolant=flow_matching.RectifiedFlow(),
    time_sampling=time_sampler,
    diagonal_sampling_rate=config.diagonal_sampling_rate,
    norm_power=config.norm_power,
    norm_regularization=config.norm_regularization,
)
trainer = flow_map.MeanFlowTrainer(
    ema_decay=config.ema_decay,
    model=mf_model,
    rng=jax.random.PRNGKey(config.seed),
    optimizer=optax.adam(config.learning_rate),
)

train_out = trainer.train(
    batch_iterator(train_data, config.batch_size, seed=config.seed + 2),
    num_steps=config.num_train_steps,
).compute()
eval_batch = {k: v[: config.batch_size] for k, v in eval_data.items()}
eval_out = trainer.eval(iter([eval_batch]), num_steps=1).compute()
print(
    f"Train loss: {float(train_out['train_loss']):.4f} | "
    f"Eval L1 loss: {float(eval_out['eval_loss']):.4f}"
)

## 5. Compare 1-, 2-, and 4-NFE Backward MeanFlow Sampling


In [ ]:
infer_fn = flow_map.MeanFlowTrainer.inference_fn_from_state_dict(
    trainer.train_state, net, use_ema=True
)
x_prior = jnp.asarray(eval_batch["x_1"])
cond_eval = {"emb:label": jnp.asarray(eval_batch["emb:label"])}

samples_by_nfe = {}
metrics = {
    "train_loss": float(train_out["train_loss"]),
    "eval_loss": float(eval_out["eval_loss"]),
}
for nfe in config.eval_num_steps:
  samples = np.asarray(
      flow_map.sample_backward_mean_flow(
          infer_fn, x_prior, num_steps=nfe, cond=cond_eval
      )
  )
  samples_by_nfe[nfe] = samples
  metrics[f"sample_std_{nfe}nfe"] = float(np.std(samples))
  metrics[f"sample_mean_{nfe}nfe"] = float(np.mean(samples))

for k, v in metrics.items():
  print(f"{k:18s}: {v:.4f}")

fig, axes = plt.subplots(1, len(config.eval_num_steps), figsize=(10, 3))
if len(config.eval_num_steps) == 1:
  axes = [axes]
for ax, nfe in zip(axes, config.eval_num_steps):
  for cls_idx in range(config.num_classes):
    mask = eval_batch["emb:label"] == cls_idx
    if np.any(mask):
      ax.plot(samples_by_nfe[nfe][mask][0, :, 0], label=f"class {cls_idx}")
  ax.set_title(f"MeanFlow ({nfe} NFE)")
  ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(workdir, "mean_flow_nfe_comparison.png"), dpi=100)
plt.show()